# Clase 5 — Introducción a las redes neuronales convolucionales

Curso **Deep Learning con Python y PyTorch desde cero** · DIH DATAlife

**Duración estimada de la clase:** 100 min

## Objetivos de la clase

- Explicar qué resuelve una convolución frente a una capa densa en visión por computador
- Describir los conceptos de filtro, kernel, stride (paso) y pooling
- Construir una red neuronal convolucional (CNN, Convolutional Neural Network) sencilla con `nn.Conv2d` y capas de pooling

Este notebook combina teoría (celdas Markdown) y código funcional en PyTorch, listo para ejecutar en Google Colab sin instalar nada adicional. Todavía no vas a entrenar la red: eso lo harás en la clase 6. Aquí construyes la arquitectura y compruebas el forward pass y las formas de los tensores. Consulta el capítulo 5 del manual del curso para la explicación completa.

## Parte 1 — Los límites de la capa densa

En la clase anterior entrenaste una red densa con `nn.Linear` sobre MNIST, y para eso tuviste que aplanar cada imagen de 28x28 píxeles en un vector de 784 valores. Al aplanar la imagen, pierdes la información sobre qué píxel está al lado de cuál: la red densa recibe una lista de números sin ninguna noción de "vecino", así que tiene que aprender desde cero, a fuerza de ejemplos, que ciertos píxeles suelen aparecer juntos formando un trazo.

Las redes neuronales convolucionales (CNN) mantienen la imagen como una cuadrícula de dos dimensiones y usan capas especializadas que sí aprovechan esa estructura espacial. Vamos a comprobarlo paso a paso, cargando primero un lote real de MNIST.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt

print('Versión de PyTorch:', torch.__version__)
print('Versión de torchvision:', torchvision.__version__)

In [ ]:
# Cargamos MNIST con torchvision, igual que en la clase 2, y lo servimos en
# lotes (batches) con DataLoader
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Tomamos un único lote de ejemplo: lo usaremos en el resto del notebook
imagenes, etiquetas = next(iter(train_loader))

print('Forma del lote de imágenes:', imagenes.shape)   # (lote, canales, alto, ancho)
print('Forma del lote de etiquetas:', etiquetas.shape)

plt.imshow(imagenes[0][0], cmap='gray')
plt.title(f'Etiqueta: {etiquetas[0].item()}')
plt.axis('off')
plt.show()

## Parte 2 — Qué es una convolución

Fíjate en la forma del lote: `(32, 1, 28, 28)`. En PyTorch, un lote de imágenes siempre sigue el orden `(lote, canales, alto, ancho)`; aquí, 32 imágenes de un canal (escala de grises) de 28x28 píxeles, sin aplanar.

Una **convolución** desliza un **filtro** (también llamado **kernel**: una pequeña matriz de números, normalmente de 3x3) por toda la imagen. En cada posición, el filtro multiplica cada uno de sus valores por el píxel que tiene debajo, suma los resultados, y ese número se convierte en el valor de un píxel en la imagen de salida (el **mapa de características**). El filtro se repite, con los mismos números, en todas las posiciones de la imagen: por eso una convolución necesita muchos menos parámetros que una capa densa equivalente.

Vamos a implementar esa idea a mano, aplicando un filtro que resalta bordes horizontales a uno de los dígitos del lote.

In [ ]:
def aplicar_filtro(imagen, filtro):
    """Aplica manualmente un filtro de convolución 3x3 (stride=1, sin padding)
    a una imagen en escala de grises representada como array 2D de numpy."""
    imagen_filtrada = np.zeros_like(imagen)
    alto, ancho = imagen.shape

    for x in range(1, alto - 1):
        for y in range(1, ancho - 1):
            suma = 0.0
            for fx in range(3):
                for fy in range(3):
                    suma += imagen[x + fx - 1, y + fy - 1] * filtro[fx][fy]
            imagen_filtrada[x, y] = suma

    return imagen_filtrada


imagen_np = imagenes[0][0].numpy()

filtro_horizontal = [[-1, -2, -1],
                      [ 0,  0,  0],
                      [ 1,  2,  1]]

imagen_filtrada = aplicar_filtro(imagen_np, filtro_horizontal)

fig, ejes = plt.subplots(1, 2, figsize=(6, 3))
ejes[0].imshow(imagen_np, cmap='gray')
ejes[0].set_title('Original')
ejes[0].axis('off')
ejes[1].imshow(imagen_filtrada, cmap='gray')
ejes[1].set_title('Tras el filtro (bordes horizontales)')
ejes[1].axis('off')
plt.show()

### Qué es el stride

En el bucle anterior, `x` e `y` avanzan de uno en uno: eso es un **stride** (paso) de uno. El stride es la distancia, en píxeles, que el filtro avanza en cada desplazamiento. Con un stride mayor (por ejemplo, dos), el filtro salta de dos en dos posiciones, examina menos posiciones de la imagen y produce un mapa de características más pequeño.

Prueba a cambiar `filtro_horizontal` por `[[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]` (bordes verticales) y vuelve a ejecutar la celda anterior: verás que se resaltan las líneas verticales del dígito en vez de las horizontales. El filtro determina qué rasgo se hace visible, pero en una CNN real esos números no los eliges tú: la red los aprende sola durante el entrenamiento, ajustándolos con retropropagación igual que ajusta los pesos de una capa `nn.Linear`.

### Pooling: resumir sin perder lo importante

El **pooling** reduce el tamaño del mapa de características conservando lo esencial de lo que el filtro ya detectó. La variante más común es **max pooling**: se recorre la imagen en bloques (por ejemplo, de 2x2) y de cada bloque solo se conserva el valor más alto.

In [ ]:
def max_pooling(imagen, tamano=2):
    """Aplica max pooling de tamano x tamano sobre una imagen en escala de grises."""
    alto, ancho = imagen.shape
    nueva_alto, nueva_ancho = alto // tamano, ancho // tamano
    imagen_reducida = np.zeros((nueva_alto, nueva_ancho))

    for x in range(0, alto - 1, tamano):
        for y in range(0, ancho - 1, tamano):
            bloque = imagen[x:x + tamano, y:y + tamano]
            imagen_reducida[x // tamano, y // tamano] = np.max(bloque)

    return imagen_reducida


imagen_pooled = max_pooling(imagen_filtrada, tamano=2)

print('Tamaño antes del pooling:', imagen_filtrada.shape)
print('Tamaño después del pooling:', imagen_pooled.shape)

plt.imshow(imagen_pooled, cmap='gray')
plt.title('Después de max pooling 2x2')
plt.axis('off')
plt.show()

La imagen resultante ocupa la cuarta parte de píxeles que la original, pero las zonas donde el filtro respondió con más intensidad siguen siendo reconocibles. Esa es la doble ventaja del pooling: menos cálculo para las siguientes capas y más robustez frente a pequeños desplazamientos del trazo en la imagen.

## Parte 3 — Construir una CNN con `nn.Conv2d` y pooling

En la práctica, no implementamos las convoluciones ni el pooling a mano: PyTorch ya incluye las capas `nn.Conv2d` y `nn.MaxPool2d`, mucho más rápidas porque están optimizadas en C++ y pueden aprovechar la GPU. Vamos a definir una CNN sencilla, siguiendo el mismo patrón de siempre: una clase que hereda de `nn.Module`, con las capas declaradas en `__init__` y la conectividad definida en `forward`.

In [ ]:
class CNNSimple(nn.Module):
    """CNN sencilla para clasificar MNIST (1 canal, 28x28 píxeles) en 10 clases."""

    def __init__(self):
        super().__init__()
        # Primer bloque: 1 canal de entrada (escala de grises), 16 filtros de
        # 3x3, con padding=1 para no perder tamaño en los bordes
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3,
                                stride=1, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Segundo bloque: recibe los 16 mapas de características del primer
        # bloque y aprende 32 filtros nuevos, combinando esos rasgos
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3,
                                stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Capa final densa: clasifica a partir de los mapas de características
        # ya reducidos por las dos capas de pooling (28 -> 14 -> 7)
        self.fc = nn.Linear(32 * 7 * 7, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        x = x.view(x.size(0), -1)   # aplanamos justo antes de la capa densa
        x = self.fc(x)
        return x


modelo = CNNSimple()
print(modelo)

In [ ]:
# Forward pass sobre el lote real de MNIST cargado en la Parte 1,
# imprimiendo la forma del tensor después de cada capa
x = imagenes
print('Entrada:                ', x.shape)

x = F.relu(modelo.conv1(x))
print('Tras conv1 + ReLU:       ', x.shape)

x = modelo.pool1(x)
print('Tras pool1:              ', x.shape)

x = F.relu(modelo.conv2(x))
print('Tras conv2 + ReLU:       ', x.shape)

x = modelo.pool2(x)
print('Tras pool2:              ', x.shape)

x = x.view(x.size(0), -1)
print('Tras aplanar (flatten):  ', x.shape)

x = modelo.fc(x)
print('Tras fc (salida):        ', x.shape)

## Resumen y siguientes pasos

- Una capa densa mira todos los píxeles a la vez y pierde la relación espacial entre ellos; una convolución recorre la imagen con un filtro que reutiliza los mismos pesos en toda la imagen
- El filtro (o kernel) es una pequeña matriz de números que se aprende durante el entrenamiento, no se diseña a mano
- El stride controla cuántos píxeles avanza el filtro en cada paso; un stride mayor produce una salida más pequeña
- El pooling (`nn.MaxPool2d`) reduce el tamaño de los mapas de características conservando el valor más alto de cada bloque
- Una CNN en PyTorch combina bloques `nn.Conv2d` + pooling, definidos dentro de una clase `nn.Module`, y termina con una capa `nn.Linear` cuyo tamaño de entrada depende de cómo cambió la forma del tensor en las capas anteriores

Ahora abre el notebook de práctica de esta clase (`clase05_practica_introduccion_a_redes_convolucionales.ipynb`) para completar tú mismo una CNN con un bloque convolucional adicional y para comprobar el forward pass sobre un lote sintético. En la clase 6 vas a entrenar esta misma arquitectura sobre MNIST.